### python -m pip install opencv-python ultralytics

In [1]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")

objc[98480]: Class AVFFrameReceiver is implemented in both /opt/anaconda3/envs/pupil/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1124ec3a8) and /opt/anaconda3/envs/pupil/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x13536c3a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[98480]: Class AVFAudioReceiver is implemented in both /opt/anaconda3/envs/pupil/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1124ec3f8) and /opt/anaconda3/envs/pupil/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x13536c3f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


In [3]:

import zmq
from msgpack import unpackb, packb
import numpy as np

In [4]:
import cv2

#### this is yolo webcam tutorial

In [ ]:
#this is testing with camera:

# Open webcam (0 = default camera)
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame)

    # Draw results on frame
    annotated_frame = results[0].plot()

    # Show the frame
    cv2.imshow("YOLO Webcam", annotated_frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 1 person, 52.8ms
Speed: 4.1ms preprocess, 52.8ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 bed, 39.4ms
Speed: 2.2ms preprocess, 39.4ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 bed, 36.8ms
Speed: 1.6ms preprocess, 36.8ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 45.2ms
Speed: 1.6ms preprocess, 45.2ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 bed, 34.4ms
Speed: 1.8ms preprocess, 34.4ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 2 beds, 34.6ms
Speed: 1.7ms preprocess, 34.6ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 2 beds, 36.8ms
Speed: 1.5ms preprocess, 36.8ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 2 beds, 40.0ms
Speed: 1.5ms preprocess, 40.0ms inference, 0

: 

## pupil core implementation

In [7]:
context = zmq.Context()
# open a req port to talk to pupil
addr = '127.0.0.1'  # remote ip or localhost
req_port = "50020"  # same as in the pupil remote gui
req = context.socket(zmq.REQ)
req.connect("tcp://{}:{}".format(addr, req_port))
# ask for the sub port
req.send_string('SUB_PORT')
sub_port = req.recv_string()

# send notification:
def notify(notification):
    """Sends ``notification`` to Pupil Remote"""
    topic = 'notify.' + notification['subject']
    payload = packb(notification, use_bin_type=True)
    req.send_string(topic, flags=zmq.SNDMORE)
    req.send(payload)
    return req.recv_string()

# Start frame publisher with format BGR
# notify({'subject': 'start_plugin', 'name': 'Frame_Publisher', 'args': {'format': 'bgr', 'fps': 1}})
notify({'subject': 'start_plugin', 'name': 'Frame_Publisher', 'args': {'format': 'bgr'}})

# open a sub port to listen to pupil
sub = context.socket(zmq.SUB)
sub.connect("tcp://{}:{}".format(addr, sub_port))

# set subscriptions to topics
# recv just pupil/gaze/notifications
sub.setsockopt_string(zmq.SUBSCRIBE, 'frame.')
# this is getting the gaze data
sub.setsockopt_string(zmq.SUBSCRIBE, 'gaze.')

def recv_from_sub():
    '''Recv a message with topic, payload.

    Topic is a utf-8 encoded string. Returned as unicode object.
    Payload is a msgpack serialized dict. Returned as a python dict.

    Any addional message frames will be added as a list
    in the payload dict with key: '__raw_data__' .
    '''
    topic = sub.recv_string()
    payload = unpackb(sub.recv(), encoding='utf-8')
    extra_frames = []
    while sub.get(zmq.RCVMORE):
        extra_frames.append(sub.recv())
    if extra_frames:
        payload['__raw_data__'] = extra_frames
    return topic, payload


recent_world = None
recent_world_meta = None
recent_gaze = None

chatgpt corrected ver

In [ ]:
# keep latest gaze always
latest_gaze = None

while True:
    topic, msg = recv_from_sub()

    if topic.startswith("gaze") and "norm_pos" in msg:
        latest_gaze = {
            "timestamp": msg.get("timestamp"),
            "confidence": msg.get("confidence", 0.0),
            "norm_pos": msg["norm_pos"],
        }

    # only infer when a new frame arrives
    if topic == "frame.world" and latest_gaze is not None:
        frame = np.frombuffer(msg['__raw_data__'][0], dtype=np.uint8).reshape(msg['height'], msg['width'], 3)

        results = model(frame)
        annotated = results[0].plot()

        cv2.imshow("YOLO Webcam", annotated)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cv2.destroyAllWindows()


### cooked version so far with displaying images

In [10]:
while True:
    topic, msg = recv_from_sub()

    
    # world frame
    if topic == 'frame.world':
        recent_world = {"timestamp": msg.get("timestamp"),
            "bgr": np.frombuffer(msg['__raw_data__'][0], dtype=np.uint8).reshape(msg['height'], msg['width'], 3),
            }
    # gaze
    elif topic.startswith("gaze"):
        # gaze is in the dict itself
        if "norm_pos" in msg:
            recent_gaze = {
                "timestamp": msg.get("timestamp"),
                "confidence": msg.get("confidence", 0.0),
                "norm_pos": msg["norm_pos"],  # [x_norm, y_norm]
            }
    

    if recent_world is not None and recent_gaze is not None:
        x_norm, y_norm = recent_gaze["norm_pos"]
        conf = recent_gaze["confidence"]

        h, w = recent_world["bgr"].shape[:2] #this is the height and width of the frame
        px = int(x_norm * w)
        py = int((1.0 - y_norm) * h)  # flip Y if needed, this is converting the norm 2d pose of the eyegaze interms of pixel data

        print(recent_world["timestamp"])
        print(recent_gaze["timestamp"])
        print()

        results = model(recent_world["bgr"], show = True)
        
        # # Draw results on frame
        # annotated_frame = results[0].plot()

        # # Show the frame
        # cv2.imshow("YOLO Webcam", annotated_frame)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

#end the monitor screen
cv2.destroyAllWindows()

563370.850085
563370.821438




0: 384x640 1 person, 1 bottle, 6 chairs, 1 laptop, 1599.9ms
Speed: 71.8ms preprocess, 1599.9ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)
563370.850085
563370.821438


0: 384x640 1 person, 1 bottle, 6 chairs, 1 laptop, 358.2ms
Speed: 17.0ms preprocess, 358.2ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
563370.850085
563370.821438


0: 384x640 1 person, 1 bottle, 6 chairs, 1 laptop, 289.3ms
Speed: 5.5ms preprocess, 289.3ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
563370.850085
563370.829507


0: 384x640 1 person, 1 bottle, 6 chairs, 1 laptop, 220.3ms
Speed: 5.9ms preprocess, 220.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
563370.850085
563370.837576


0: 384x640 1 person, 1 bottle, 6 chairs, 1 laptop, 241.2ms
Speed: 16.5ms preprocess, 241.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
563370.850085
563370.845645


0: 384x640 1 person, 1 bottle, 6 chairs, 1 laptop, 317.2ms
S

## urm later

In [ ]:
while True:
    topic, msg = recv_from_sub()

    
    # world frame
    if topic == 'frame.world':
        recent_world = {"timestamp": msg.get("timestamp"),
            "bgr": np.frombuffer(msg['__raw_data__'][0], dtype=np.uint8).reshape(msg['height'], msg['width'], 3),
            }
    # gaze
    elif topic.startswith("gaze"):
        # gaze is in the dict itself
        if "norm_pos" in msg:
            recent_gaze = {
                "timestamp": msg.get("timestamp"),
                "confidence": msg.get("confidence", 0.0),
                "norm_pos": msg["norm_pos"],  # [x_norm, y_norm]
            }
    

    if recent_world is not None and recent_gaze is not None:
        x_norm, y_norm = recent_gaze["norm_pos"]
        conf = recent_gaze["confidence"]

        h, w = recent_world["bgr"].shape[:2] #this is the height and width of the frame
        px = int(x_norm * w)
        py = int((1.0 - y_norm) * h)  # flip Y if needed, this is converting the norm 2d pose of the eyegaze interms of pixel data

        print(recent_world["timestamp"])
        print(recent_gaze["timestamp"])
        print()

        results = model(recent_world["bgr"])
        
        for result in results:
            boxes = result.boxes

            if boxes is not None:
                for box in boxes:
                    print("Class ID:", int(box.cls))
                    print("Confidence:", float(box.conf))
                    print("Bounding box:", box.xyxy.tolist())
        
         # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

#end the monitor screen
cv2.destroyAllWindows()

562495.554096
562495.507321


0: 384x640 1 person, 1 bottle, 5 chairs, 1 dining table, 1 laptop, 46.3ms
Speed: 2.3ms preprocess, 46.3ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)
Class ID: 63
Confidence: 0.9173451662063599
Bounding box: [[455.0428161621094, 269.5723876953125, 1031.165283203125, 713.5277709960938]]
Class ID: 56
Confidence: 0.8554492592811584
Bounding box: [[108.71995544433594, 304.61956787109375, 321.9355773925781, 573.3363037109375]]
Class ID: 56
Confidence: 0.8102485537528992
Bounding box: [[643.3800048828125, 151.7166748046875, 836.3167724609375, 277.36785888671875]]
Class ID: 0
Confidence: 0.723456621170044
Bounding box: [[275.6732177734375, 5.084320068359375, 504.4405822753906, 381.724853515625]]
Class ID: 39
Confidence: 0.6707693338394165
Bounding box: [[1185.3349609375, 227.8509521484375, 1245.1522216796875, 378.5089416503906]]
Class ID: 56
Confidence: 0.6311498284339905
Bounding box: [[1028.02587890625, 184.38116455078125, 1179.221923828125

NameError: name 'cap' is not defined